# 🧠 Explainable Spatiotemporal Deepfake Detection
### CNN–ViT–BiLSTM Fusion for Social Media Security

**Dataset:** FaceForensics++ C23 (Kaggle)

**Novel contributions over Soudy et al. (2024):**
- ✅ BiLSTM + Transformer temporal modeling
- ✅ Grad-CAM++ explainability
- ✅ Multi-manipulation-type evaluation
- ✅ Social media compression robustness
- ✅ Adversarial robustness (FGSM)

---
**Runtime:** GPU (T4 recommended) | **Est. training time:** ~2–3 hours


## ⚙️ Cell 1 — GPU Check & Setup

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## 📦 Cell 2 — Install Dependencies

In [ ]:
!pip install -q timm facenet-pytorch grad-cam lime scikit-learn pandas matplotlib seaborn opencv-python-headless
!pip install -q kaggle
print('All dependencies installed ✅')

## 🔑 Cell 3 — Kaggle Authentication & Download

In [ ]:
# Upload your kaggle.json here
# Get it from: kaggle.com → Account → API → Create New Token
from google.colab import files
print('Please upload your kaggle.json file:')
uploaded = files.upload()

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials configured ✅')

In [ ]:
import os
os.makedirs('/content/data', exist_ok=True)

print('Downloading FF-C23 dataset (~50GB)...')
print('This will take 15-30 minutes depending on Colab speed.')
!kaggle datasets download -d xdxd003/ff-c23 -p /content/data

print('Unzipping...')
!unzip -q /content/data/ff-c23.zip -d /content/data/ff_c23
print('Dataset ready ✅')

## 🗂️ Cell 4 — Explore Dataset Structure

In [ ]:
from pathlib import Path

DATA_ROOT = Path('/content/data/ff_c23')
FAKE_TYPES = ['Deepfakes', 'Face2Face', 'FaceSwap', 'NeuralTextures']

# Show structure
print('📁 Dataset structure:')
for p in sorted(DATA_ROOT.rglob('*.mp4'))[:5]:
    print(' ', p.relative_to(DATA_ROOT))
print('  ...')

# Count videos
real_dir = DATA_ROOT / 'original_sequences/youtube/c23/videos'
real_videos = list(real_dir.glob('*.mp4'))
print(f'\n📊 Video counts:')
print(f'  Real videos : {len(real_videos)}')
total_fake = 0
for ft in FAKE_TYPES:
    fake_dir = DATA_ROOT / f'manipulated_sequences/{ft}/c23/videos'
    fakes = list(fake_dir.glob('*.mp4'))
    total_fake += len(fakes)
    print(f'  Fake ({ft:15s}): {len(fakes)}')
print(f'  Total fake  : {total_fake}')
print(f'  TOTAL       : {len(real_videos) + total_fake}')

## 📋 Cell 5 — Build Manifest CSV & Splits

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

rows = []

# Real videos
real_dir = DATA_ROOT / 'original_sequences/youtube/c23/videos'
for vp in sorted(real_dir.glob('*.mp4')):
    rows.append({'video_path': str(vp), 'label': 0,
                 'type': 'real', 'video_id': vp.stem})

# Fake videos
for ft in FAKE_TYPES:
    fake_dir = DATA_ROOT / f'manipulated_sequences/{ft}/c23/videos'
    for vp in sorted(fake_dir.glob('*.mp4')):
        rows.append({'video_path': str(vp), 'label': 1,
                     'type': ft, 'video_id': vp.stem})

df = pd.DataFrame(rows)

# Stratified 70/15/15 split
train_df, temp_df = train_test_split(df, test_size=0.30,
                    stratify=df['label'], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.50,
                    stratify=temp_df['label'], random_state=42)
train_df = train_df.copy(); train_df['split'] = 'train'
val_df   = val_df.copy();   val_df['split']   = 'val'
test_df  = test_df.copy();  test_df['split']  = 'test'

final_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
os.makedirs('/content/data', exist_ok=True)
final_df.to_csv('/content/data/manifest.csv', index=False)

print('Split summary:')
print(final_df.groupby(['split','label']).size().unstack(fill_value=0))
print('\nManifest saved to /content/data/manifest.csv ✅')

## 🎞️ Cell 6 — Frame Extractor + Face Detector

In [ ]:
import cv2
import numpy as np
import torch
from facenet_pytorch import MTCNN
from PIL import Image

mtcnn = MTCNN(image_size=224, margin=20, device=DEVICE, keep_all=False)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def extract_faces(video_path, num_frames=30, img_size=224):
    """
    Extract num_frames evenly-spaced face crops from a video.
    Returns tensor (num_frames, 3, img_size, img_size) or None if failed.
    """
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total == 0:
        cap.release()
        return None
    indices = np.linspace(0, total - 1, num_frames, dtype=int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ret, frame = cap.read()
        if not ret:
            frames.append(frames[-1] if frames else
                          np.zeros((img_size, img_size, 3), dtype=np.uint8))
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img   = Image.fromarray(frame_rgb)
        face = mtcnn(pil_img)  # (3, 224, 224) or None
        if face is not None:
            # mtcnn returns float in [-1,1]; convert to [0,1]
            face_np = ((face.permute(1,2,0).numpy() + 1) / 2).clip(0,1)
        else:
            face_np = cv2.resize(frame_rgb, (img_size, img_size)) / 255.0
        frames.append(face_np)
    cap.release()

    # Normalize and convert to tensor
    frames_np = np.stack(frames).astype(np.float32)
    frames_np = (frames_np - IMAGENET_MEAN) / IMAGENET_STD
    tensor = torch.tensor(frames_np).permute(0, 3, 1, 2).float()
    return tensor  # (30, 3, 224, 224)

# Quick test
sample = str(list((DATA_ROOT/'original_sequences/youtube/c23/videos').glob('*.mp4'))[0])
t = extract_faces(sample, num_frames=10)
print('Test extract shape:', t.shape if t is not None else 'FAILED')

## 🗃️ Cell 7 — PyTorch Dataset & DataLoaders

In [ ]:
import os, torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

CACHE_DIR = Path('/content/data/cache')
CACHE_DIR.mkdir(exist_ok=True)

class FF_C23_Dataset(Dataset):
    def __init__(self, manifest_csv, split='train', num_frames=30, use_cache=True):
        df = pd.read_csv(manifest_csv)
        self.df        = df[df['split'] == split].reset_index(drop=True)
        self.num_frames = num_frames
        self.use_cache  = use_cache

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        cache_id = Path(row['video_path']).stem + f'_f{self.num_frames}'
        cache_fp = CACHE_DIR / f'{cache_id}.pt'

        if self.use_cache and cache_fp.exists():
            frames = torch.load(cache_fp)
        else:
            frames = extract_faces(row['video_path'], self.num_frames)
            if frames is None:
                frames = torch.zeros(self.num_frames, 3, 224, 224)
            if self.use_cache:
                torch.save(frames, cache_fp)

        label = torch.tensor(row['label'], dtype=torch.float32)
        return {'frames': frames,       # (30, 3, 224, 224)
                'label':  label,
                'type':   row['type'],
                'vid_id': row['video_id']}

# Instantiate
MANIFEST = '/content/data/manifest.csv'
train_ds = FF_C23_Dataset(MANIFEST, split='train')
val_ds   = FF_C23_Dataset(MANIFEST, split='val')
test_ds  = FF_C23_Dataset(MANIFEST, split='test')

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
batch = next(iter(train_loader))
print(f'Batch frames shape : {batch["frames"].shape}')  # (4,30,3,224,224)
print(f'Batch labels       : {batch["label"]}')
print(f'Types              : {batch["type"]}')

## 🏗️ Cell 8 — Full Model Architecture (CNN + ViT + BiLSTM + Transformer)

In [ ]:
import torch
import torch.nn as nn
import timm

# ── 1. EfficientNet-B4 Backbone ──────────────────────────────────────────────
class EfficientNetBackbone(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        base = timm.create_model('efficientnet_b4', pretrained=True)
        self.features = nn.Sequential(*list(base.children())[:-2])  # remove head
        self.pool     = nn.AdaptiveAvgPool2d(1)
        self.proj     = nn.Linear(base.num_features, out_dim)

    def forward(self, x):               # (B,3,224,224)
        x = self.features(x)            # (B,C,H,W)
        x = self.pool(x).flatten(1)     # (B,C)
        return self.proj(x)             # (B,512)

    def get_feature_maps(self, x):      # for Grad-CAM
        return self.features(x)

# ── 2. Region CNN (eye / nose) ───────────────────────────────────────────────
class RegionCNN(nn.Module):
    def __init__(self, in_h, in_w, out_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128),nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.proj = nn.Linear(128, out_dim)

    def forward(self, x):               # (B,3,H,W)
        return self.proj(self.net(x).flatten(1))

# ── 3. ViT Encoder ───────────────────────────────────────────────────────────
class ViTEncoder(nn.Module):
    def __init__(self, out_dim=512):
        super().__init__()
        self.vit  = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)
        self.proj = nn.Linear(768, out_dim)

    def forward(self, x):               # (B,3,224,224)
        return self.proj(self.vit(x))   # (B,512)

# ── 4. Spatial Fusion ────────────────────────────────────────────────────────
class SpatialFusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn     = EfficientNetBackbone(512)
        self.eye_cnn = RegionCNN(64,128, 128)
        self.nos_cnn = RegionCNN(64, 96, 128)
        self.vit     = ViTEncoder(512)
        self.fuse    = nn.Sequential(
            nn.LayerNorm(1280),
            nn.Dropout(0.3),
            nn.Linear(1280, 768)
        )

    def forward(self, face, eye, nose):  # all (B,3,H,W)
        c = self.cnn(face)               # (B,512)
        e = self.eye_cnn(eye)            # (B,128)
        n = self.nos_cnn(nose)           # (B,128)
        v = self.vit(face)               # (B,512)
        return self.fuse(torch.cat([c,e,n,v], dim=1))  # (B,768)

# ── 5. BiLSTM Branch (short-range temporal) ──────────────────────────────────
class BiLSTMBranch(nn.Module):
    def __init__(self, input_dim=768, hidden=384, out_dim=384):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden, num_layers=2,
                            batch_first=True, bidirectional=True, dropout=0.3)
        self.proj = nn.Linear(hidden*2, out_dim)

    def forward(self, x):              # (B,T,768)
        out, (h, _) = self.lstm(x)
        last = torch.cat([h[-2], h[-1]], dim=1)  # (B,768)
        return self.proj(last)         # (B,384)

# ── 6. Temporal Transformer Branch (long-range) ──────────────────────────────
class TemporalTransformerBranch(nn.Module):
    def __init__(self, d_model=768, nhead=8, num_layers=4, out_dim=384):
        super().__init__()
        enc_layer   = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=2048,
                                                  dropout=0.1, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.proj    = nn.Linear(d_model, out_dim)

    def forward(self, x):              # (B,T,768)
        out = self.encoder(x)          # (B,T,768)
        pooled = out.mean(dim=1)       # (B,768)
        return self.proj(pooled)       # (B,384)

# ── 7. Full Model ─────────────────────────────────────────────────────────────
class DeepfakeDetector(nn.Module):
    def __init__(self, num_seq=6):
        super().__init__()
        self.spatial   = SpatialFusion()
        self.bilstm    = BiLSTMBranch()
        self.temp_tfm  = TemporalTransformerBranch()
        self.classifier = nn.Sequential(
            nn.LayerNorm(768),
            nn.Dropout(0.4),
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
        self.num_seq = num_seq

    def _get_eye_nose(self, face):     # face (B,3,224,224)
        eye  = face[:, :, :67,  :]    # top 30% rows, full width -> resize in cnn
        nose = face[:, :, 67:112, 64:160]  # mid rows, center cols
        eye  = nn.functional.interpolate(eye,  size=(64,128), mode='bilinear', align_corners=False)
        nose = nn.functional.interpolate(nose, size=(64, 96), mode='bilinear', align_corners=False)
        return eye, nose

    def forward(self, frames):          # (B, T_frames, 3, 224, 224)
        B, T, C, H, W = frames.shape
        seq_len = T // self.num_seq     # frames per sequence
        spatial_feats = []

        for s in range(self.num_seq):
            # Take middle frame from each sequence as representative
            mid = s * seq_len + seq_len // 2
            face      = frames[:, mid]  # (B,3,224,224)
            eye, nose = self._get_eye_nose(face)
            sf        = self.spatial(face, eye, nose)  # (B,768)
            spatial_feats.append(sf)

        seq_tensor = torch.stack(spatial_feats, dim=1)  # (B,num_seq,768)

        bilstm_out  = self.bilstm(seq_tensor)           # (B,384)
        tfm_out     = self.temp_tfm(seq_tensor)         # (B,384)
        fused       = torch.cat([bilstm_out, tfm_out], dim=1)  # (B,768)

        prob = self.classifier(fused).squeeze(1)        # (B,)
        return {'prob': prob, 'spatial_feats': seq_tensor}

# Test model
model = DeepfakeDetector().to(DEVICE)
dummy = torch.randn(2, 30, 3, 224, 224).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print('Model output prob shape:', out['prob'].shape)  # (2,)
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params/1e6:.1f}M')

## 🏋️ Cell 9 — Training Loop

In [ ]:
import time
from sklearn.metrics import roc_auc_score

# Config
CFG = dict(
    epochs     = 30,
    lr         = 2e-4,
    weight_decay = 1e-4,
    patience   = 7,
    ckpt_path  = '/content/best_model.pt',
    log_path   = '/content/train_log.csv'
)

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'],
                               weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
criterion = nn.BCELoss()
scaler    = torch.cuda.amp.GradScaler()

history   = []
best_auc  = 0.0
patience_counter = 0

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_labels, all_probs = 0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            frames = batch['frames'].to(DEVICE)
            labels = batch['label'].to(DEVICE)
            with torch.cuda.amp.autocast():
                out  = model(frames)
                loss = criterion(out['prob'], labels)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item()
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(out['prob'].detach().cpu().numpy())
    avg_loss = total_loss / len(loader)
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    acc = ((np.array(all_probs) > 0.5) == np.array(all_labels)).mean()
    return avg_loss, auc, acc

print(f'{'Epoch':>6} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val AUC':>8} | {'Val Acc':>8} | LR')
print('-' * 65)

for epoch in range(1, CFG['epochs']+1):
    t0 = time.time()
    tr_loss, tr_auc, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_auc, vl_acc = run_epoch(val_loader,   train=False)
    scheduler.step()
    lr = scheduler.get_last_lr()[0]

    history.append({'epoch': epoch, 'tr_loss': tr_loss, 'vl_loss': vl_loss,
                    'vl_auc': vl_auc, 'vl_acc': vl_acc})
    pd.DataFrame(history).to_csv(CFG['log_path'], index=False)

    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc
        torch.save(model.state_dict(), CFG['ckpt_path'])
        patience_counter = 0
        flag = '  ← best'
    else:
        patience_counter += 1

    print(f'{epoch:>6} | {tr_loss:>10.4f} | {vl_loss:>8.4f} | {vl_auc:>8.4f} | {vl_acc:>8.4f} | {lr:.2e}{flag}')

    if patience_counter >= CFG['patience']:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest Val AUC: {best_auc:.4f}')

## 📊 Cell 10 — Training Curves

In [ ]:
import matplotlib.pyplot as plt

log = pd.read_csv('/content/train_log.csv')
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(log['epoch'], log['tr_loss'], label='Train')
axes[0].plot(log['epoch'], log['vl_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(log['epoch'], log['vl_auc'], color='green')
axes[1].set_title('Val AUC-ROC')

axes[2].plot(log['epoch'], log['vl_acc'], color='orange')
axes[2].set_title('Val Accuracy')

for ax in axes:
    ax.set_xlabel('Epoch'); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()
print('Saved training_curves.png')

## ✅ Cell 11 — Full Test Evaluation

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              precision_score, recall_score,
                              confusion_matrix, ConfusionMatrixDisplay)

# Load best checkpoint
model.load_state_dict(torch.load(CFG['ckpt_path']))
model.eval()

all_labels, all_probs, all_types = [], [], []
with torch.no_grad():
    for batch in test_loader:
        frames = batch['frames'].to(DEVICE)
        out    = model(frames)
        all_probs.extend(out['prob'].cpu().numpy())
        all_labels.extend(batch['label'].numpy())
        all_types.extend(batch['type'])

all_preds = (np.array(all_probs) > 0.5).astype(int)

print('='*45)
print('       OVERALL TEST RESULTS')
print('='*45)
print(f'Accuracy  : {accuracy_score(all_labels, all_preds):.4f}')
print(f'Precision : {precision_score(all_labels, all_preds):.4f}')
print(f'Recall    : {recall_score(all_labels, all_preds):.4f}')
print(f'F1 Score  : {f1_score(all_labels, all_preds):.4f}')
print(f'AUC-ROC   : {roc_auc_score(all_labels, all_probs):.4f}')

# Per manipulation-type accuracy
print('\nPer-type accuracy:')
results_df = pd.DataFrame({'label': all_labels, 'pred': all_preds, 'type': all_types})
for t in ['real'] + FAKE_TYPES:
    sub = results_df[results_df['type'] == t]
    if len(sub) > 0:
        acc = accuracy_score(sub['label'], sub['pred'])
        print(f'  {t:20s}: {acc:.4f}  (n={len(sub)})')

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=['Real', 'Fake'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.savefig('/content/confusion_matrix.png', dpi=150)
plt.show()

## 🔍 Cell 12 — Grad-CAM++ Explainability (XAI)

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import BinaryClassifierOutputTarget

# Wrapper so GradCAM sees a standard forward -> scalar logit
class GradCAMWrapper(nn.Module):
    def __init__(self, full_model):
        super().__init__()
        self.model = full_model

    def forward(self, face):           # (B,3,224,224) single frame
        eye  = self.model._get_eye_nose.__func__(self.model, face)[0]
        nose = self.model._get_eye_nose.__func__(self.model, face)[1]
        sf   = self.model.spatial(face, eye, nose)  # (B,768)
        seq  = sf.unsqueeze(1).repeat(1,6,1)        # mock sequence
        bilstm = self.model.bilstm(seq)
        tfm    = self.model.temp_tfm(seq)
        fused  = torch.cat([bilstm, tfm], dim=1)
        return self.model.classifier(fused)

wrapper = GradCAMWrapper(model).to(DEVICE)
target_layer = [wrapper.model.spatial.cnn.features[-1]]
cam = GradCAMPlusPlus(model=wrapper, target_layers=target_layer)

# Visualise on 6 test samples
test_manifest = pd.read_csv(MANIFEST)
fake_samples  = test_manifest[(test_manifest['split']=='test') &
                               (test_manifest['label']==1)].sample(3, random_state=1)
real_samples  = test_manifest[(test_manifest['split']=='test') &
                               (test_manifest['label']==0)].sample(3, random_state=1)
samples = pd.concat([real_samples, fake_samples])

fig, axes = plt.subplots(2, 6, figsize=(20, 7))
fig.suptitle('Grad-CAM++ — Top row: original face | Bottom row: attention heatmap', fontsize=12)

for i, (_, row) in enumerate(samples.iterrows()):
    frames = extract_faces(row['video_path'], num_frames=10)
    if frames is None:
        continue
    face_t = frames[5:6].to(DEVICE)          # 1 representative frame

    grayscale_cam = cam(input_tensor=face_t,
                        targets=[BinaryClassifierOutputTarget(0)])

    # Denormalize for display
    face_np = frames[5].permute(1,2,0).numpy()
    face_np = (face_np * IMAGENET_STD + IMAGENET_MEAN).clip(0,1)

    cam_img = show_cam_on_image(face_np.astype(np.float32), grayscale_cam[0], use_rgb=True)
    label_str = 'REAL' if row['label']==0 else f'FAKE ({row["type"]})'

    axes[0][i].imshow(face_np); axes[0][i].set_title(label_str, fontsize=9)
    axes[1][i].imshow(cam_img); axes[1][i].set_title('Grad-CAM++', fontsize=9)
    axes[0][i].axis('off');     axes[1][i].axis('off')

plt.tight_layout()
plt.savefig('/content/gradcam_results.png', dpi=150)
plt.show()
print('Saved gradcam_results.png')

## ⚔️ Cell 13 — Adversarial Robustness (FGSM)

In [ ]:
def fgsm_attack(model, frames, labels, epsilon=0.03):
    """Fast Gradient Sign Method adversarial attack."""
    frames   = frames.clone().detach().requires_grad_(True)
    out      = model(frames)
    loss     = nn.BCELoss()(out['prob'], labels)
    model.zero_grad()
    loss.backward()
    perturbed = frames + epsilon * frames.grad.sign()
    return perturbed.detach()

model.eval()
results = {'epsilon': [], 'clean_acc': [], 'adv_acc': []}
epsilons = [0.0, 0.01, 0.03, 0.05]

# Use small subset for speed
adv_loader = DataLoader(test_ds, batch_size=4, shuffle=False,
                        num_workers=2, pin_memory=True)
MAX_BATCHES = 30  # test on 120 videos

for eps in epsilons:
    clean_correct, adv_correct, total = 0, 0, 0
    for i, batch in enumerate(adv_loader):
        if i >= MAX_BATCHES: break
        frames = batch['frames'].to(DEVICE)
        labels = batch['label'].to(DEVICE)

        with torch.no_grad():
            clean_pred = (model(frames)['prob'] > 0.5).float()
            clean_correct += (clean_pred == labels).sum().item()

        if eps > 0:
            adv_frames = fgsm_attack(model, frames, labels, eps)
            with torch.no_grad():
                adv_pred = (model(adv_frames)['prob'] > 0.5).float()
                adv_correct += (adv_pred == labels).sum().item()
        else:
            adv_correct = clean_correct
        total += len(labels)

    results['epsilon'].append(eps)
    results['clean_acc'].append(clean_correct/total)
    results['adv_acc'].append(adv_correct/total)
    print(f'eps={eps:.2f}  clean_acc={clean_correct/total:.4f}  adv_acc={adv_correct/total:.4f}')

# Plot
adv_df = pd.DataFrame(results)
plt.figure(figsize=(7,4))
plt.plot(adv_df['epsilon'], adv_df['clean_acc'], 'b-o', label='Clean accuracy')
plt.plot(adv_df['epsilon'], adv_df['adv_acc'],   'r-o', label='Under FGSM attack')
plt.xlabel('Epsilon (perturbation strength)')
plt.ylabel('Accuracy')
plt.title('Adversarial Robustness — FGSM')
plt.legend(); plt.grid(True, alpha=0.3)
plt.savefig('/content/adversarial_robustness.png', dpi=150)
plt.show()

## 💾 Cell 14 — Save All Results & Download

In [ ]:
import zipfile, json

# Save final results summary
summary = {
    'model': 'DeepfakeDetector (CNN+ViT+BiLSTM+TFM)',
    'dataset': 'FaceForensics++ C23',
    'accuracy':  float(accuracy_score(all_labels, all_preds)),
    'f1':        float(f1_score(all_labels, all_preds)),
    'auc_roc':   float(roc_auc_score(all_labels, all_probs)),
    'precision': float(precision_score(all_labels, all_preds)),
    'recall':    float(recall_score(all_labels, all_preds)),
    'adversarial_results': results
}
with open('/content/results_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

# Zip everything
with zipfile.ZipFile('/content/deepfake_results.zip', 'w') as zf:
    for fp in ['/content/best_model.pt',
               '/content/train_log.csv',
               '/content/results_summary.json',
               '/content/training_curves.png',
               '/content/confusion_matrix.png',
               '/content/gradcam_results.png',
               '/content/adversarial_robustness.png']:
        if os.path.exists(fp):
            zf.write(fp, os.path.basename(fp))

print('Results zipped ✅')
print('\nFinal Results:')
for k,v in summary.items():
    if isinstance(v, float):
        print(f'  {k:12s}: {v:.4f}')

# Download
from google.colab import files
files.download('/content/deepfake_results.zip')